# M3.S5 — Asynchronous Practice 3
## Part 1 — Guided Jupyter Lab
### Profile → explain → change ONE thing → remeasure

**Guided route: about 55–60 minutes**

This lab consolidates the four live sessions in Module 3:

```text
M3.S1  scalability & performance evidence
          ↓
M3.S2  memory hierarchy & locality
          ↓
M3.S3  profiling & performance analysis
          ↓
M3.S4  libraries, I/O & reproducibility
          ↓
M3.S5  evidence-led optimisation
```

You will work with one scientific mini-application and follow the complete performance-engineering workflow:

> **CORRECTNESS → BASELINE → PROFILE → HYPOTHESIS → CHANGE ONE FACTOR → REMEASURE → EXPLAIN → RECORD**

The guided optimisation changes **only loop traversal order**. The numerical method, input, compiler optimisation level, Slurm resources and output remain the same.

After this notebook, **Part 2 in Blackboard** asks you to perform the same evidence chain independently through SSH + Slurm.

> **Notebook build: M3S5-2026-09-28-v1**


## 1 — The mini-application: same mathematics, different memory access

The application repeatedly transforms a 2-D field stored in a normal C row-major array.

Two mathematically equivalent implementations are supplied.

### Baseline traversal

```c
for (j = 1; j < n-1; ++j)
    for (i = 1; i < n-1; ++i)
        out[i*n+j] = ...
```

### Locality-friendly traversal

```c
for (i = 1; i < n-1; ++i)
    for (j = 1; j < n-1; ++j)
        out[i*n+j] = ...
```

### PREDICT

1. Which traversal follows contiguous elements in a row-major C array?
2. Which should have better spatial locality?
3. Is that prediction enough to claim it is faster?

<details>
<summary><strong>Reveal</strong></summary>

The `i`-outer / `j`-inner traversal follows contiguous memory locations.

But prediction is not evidence. We still need correctness, repeated timings and profiling evidence.

</details>


In [ ]:
from pathlib import Path
import os, re, subprocess, time, shutil, statistics, json, platform

N = 1536
STEPS = 80
REPEATS = 3
OMP_THREADS = [1, 2, 4, 8]

print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("Visible CPUs:", os.cpu_count())
print("sbatch:", shutil.which("sbatch"))
print("gcc:", shutil.which("gcc"))
print("gprof:", shutil.which("gprof"))
print("perf:", shutil.which("perf"))
print("strace:", shutil.which("strace"))
print("/usr/bin/time:", Path("/usr/bin/time").exists())
print()
print("Benchmark N:", N)
print("Steps:", STEPS)
print("Repetitions:", REPEATS)


## 2 — Real C code with phase timers

The next cell writes the actual C program.

It reports timings for:

```text
initialise
compute
checkpoint
total
```

and prints a checksum.

This gives us a **timer fallback** even if a preferred profiler is unavailable.

The checkpoint is small and written once. It gives the application a real I/O phase without making I/O the intended bottleneck.


In [ ]:
app_src = r'''
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#ifdef _OPENMP
#include <omp.h>
#endif

static double now_s(void) {
#ifdef _OPENMP
    return omp_get_wtime();
#else
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec / 1e9;
#endif
}

static void parse(int argc, char **argv, int *n, int *steps,
                  int *good, const char **checkpoint) {
    *n = 1536;
    *steps = 80;
    *good = 0;
    *checkpoint = "m3s5_checkpoint.bin";

    for (int k = 1; k < argc; ++k) {
        if (!strcmp(argv[k], "--demo")) {
            *n = 41;
            *steps = 6;
        } else if (!strcmp(argv[k], "--n") && k+1 < argc) {
            *n = atoi(argv[++k]);
        } else if (!strcmp(argv[k], "--steps") && k+1 < argc) {
            *steps = atoi(argv[++k]);
        } else if (!strcmp(argv[k], "--mode") && k+1 < argc) {
            const char *m = argv[++k];
            *good = !strcmp(m, "good");
        } else if (!strcmp(argv[k], "--checkpoint") && k+1 < argc) {
            *checkpoint = argv[++k];
        }
    }
}

__attribute__((noinline))
static void initialise(double *a, double *b, int n) {
    size_t sz = (size_t)n * n;
    for (size_t k = 0; k < sz; ++k) {
        a[k] = 0.0;
        b[k] = 0.0;
    }

    for (int i = 1; i < n-1; ++i)
        for (int j = 1; j < n-1; ++j)
            a[(size_t)i*n+j] =
                (double)((i*17 + j*13) % 1000) / 1000.0;
}

__attribute__((noinline))
static void sweep_bad(const double *a, double *b, int n) {
#ifdef _OPENMP
    #pragma omp parallel for schedule(static)
#endif
    for (int j = 1; j < n-1; ++j) {
        for (int i = 1; i < n-1; ++i) {
            size_t p = (size_t)i*n+j;
            b[p] = 0.50*a[p]
                 + 0.125*a[p-1]
                 + 0.125*a[p+1]
                 + 0.125*a[p-n]
                 + 0.125*a[p+n];
        }
    }
}

__attribute__((noinline))
static void sweep_good(const double *a, double *b, int n) {
#ifdef _OPENMP
    #pragma omp parallel for schedule(static)
#endif
    for (int i = 1; i < n-1; ++i) {
        for (int j = 1; j < n-1; ++j) {
            size_t p = (size_t)i*n+j;
            b[p] = 0.50*a[p]
                 + 0.125*a[p-1]
                 + 0.125*a[p+1]
                 + 0.125*a[p-n]
                 + 0.125*a[p+n];
        }
    }
}

__attribute__((noinline))
static double checksum(const double *a, int n) {
    double sum = 0.0;
    for (int i = 1; i < n-1; ++i)
        for (int j = 1; j < n-1; ++j)
            sum += a[(size_t)i*n+j];
    return sum;
}

__attribute__((noinline))
static int write_checkpoint(const char *path, const double *a, int n) {
    FILE *f = fopen(path, "wb");
    if (!f) return 0;

    int row = n/2;
    size_t written =
        fwrite(&a[(size_t)row*n], sizeof(double), (size_t)n, f);
    fclose(f);

    return written == (size_t)n;
}

int main(int argc, char **argv) {
    int n, steps, good;
    const char *checkpoint;
    parse(argc, argv, &n, &steps, &good, &checkpoint);

    size_t sz = (size_t)n * n;
    double *a = (double*)calloc(sz, sizeof(double));
    double *b = (double*)calloc(sz, sizeof(double));
    if (!a || !b) return 2;

    double t_total0 = now_s();

    double t0 = now_s();
    initialise(a, b, n);
    double init_s = now_s() - t0;

    t0 = now_s();
    for (int s = 0; s < steps; ++s) {
        if (good) sweep_good(a, b, n);
        else      sweep_bad(a, b, n);

        double *tmp = a;
        a = b;
        b = tmp;
    }
    double compute_s = now_s() - t0;

    double sum = checksum(a, n);

    t0 = now_s();
    int io_ok = write_checkpoint(checkpoint, a, n);
    double io_s = now_s() - t0;

    double total_s = now_s() - t_total0;

#ifdef _OPENMP
    int workers = omp_get_max_threads();
#else
    int workers = 1;
#endif

    printf("PROFILE_PHASE phase=initialise seconds=%.6f\n", init_s);
    printf("PROFILE_PHASE phase=compute seconds=%.6f\n", compute_s);
    printf("PROFILE_PHASE phase=checkpoint seconds=%.6f\n", io_s);
    printf("RESULT mode=%s workers=%d n=%d steps=%d total=%.6f compute=%.6f checksum=%.12e io_ok=%d\n",
           good ? "good" : "bad", workers, n, steps,
           total_s, compute_s, sum, io_ok);

    free(a);
    free(b);
    return io_ok ? 0 : 3;
}
'''

Path("m3s5_opt_lab.c").write_text(app_src)
print("Wrote m3s5_opt_lab.c")


## 3 — Correctness before performance

Run a tiny case for both loop orders.

The checksums must agree.

This Jupyter-side run is a **correctness check only**. The real performance experiment runs through Slurm.


In [ ]:
p = subprocess.run(
    ["bash", "-lc",
     "gcc -O3 -std=gnu11 m3s5_opt_lab.c -o m3s5_opt_lab"],
    capture_output=True, text=True
)
print(p.stdout)
if p.stderr:
    print(p.stderr)
assert p.returncode == 0, "Compilation failed"

demo = {}
for mode in ("bad", "good"):
    p = subprocess.run(
        ["./m3s5_opt_lab", "--demo", "--mode", mode,
         "--checkpoint", f"demo_{mode}.bin"],
        capture_output=True, text=True
    )
    print(f"--- {mode} ---")
    print(p.stdout)
    assert p.returncode == 0
    m = re.search(r"checksum=([0-9.eE+-]+)", p.stdout)
    assert m
    demo[mode] = float(m.group(1))

delta = abs(demo["bad"] - demo["good"])
print("Checksum difference:", delta)
print("Correctness:", "PASS" if delta < 1e-9 else "FAIL")
assert delta < 1e-9


## 4 — Baseline + profile on the real cluster

The controlled baseline is:

```text
mode = bad
N = 1536
steps = 80
CPU threads = 1
compiler = GCC -O3
repetitions = 3
```

The Slurm job records the configuration and uses several evidence routes:

- phase timers — always available;
- GNU `time -v` — when available;
- `gprof` — function-level profile;
- `perf stat` — optional hardware counters if permitted;
- `strace -c` — optional system-call summary.

A missing optional tool must **not** block the lab.


In [ ]:
def clean_submit_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit(script):
    if not shutil.which("sbatch"):
        raise RuntimeError(
            "sbatch is unavailable. Run this notebook on SciTech JupyterHub."
        )
    p = subprocess.run(
        ["sbatch", "--parsable", script],
        capture_output=True, text=True, env=clean_submit_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    job = p.stdout.strip().split(";")[0]
    print("Submitted:", job)
    return job

def wait_job(job, timeout=420):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            time.sleep(2)
            return True
        print("Job", job, state)
        time.sleep(4)
    print("Timed out waiting for job", job)
    return False

def show_output(job, prefix):
    path = Path(f"{prefix}-{job}.out")
    for _ in range(10):
        if path.exists():
            txt = path.read_text(errors="replace")
            print(txt)
            return txt
        time.sleep(1)
    print("Output not found:", path)
    return ""

print("Slurm helpers ready")


In [ ]:
baseline_script = f'''#!/bin/bash
#SBATCH --job-name=m3s5_base
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=m3s5_base-%j.out

set -euo pipefail
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== CONFIGURATION ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "HOST=$(hostname)"
echo "DATE=$(date -Is)"
echo "CPU_MODEL=$(lscpu | awk -F: '/Model name/ {{gsub(/^[ \\t]+/, "", $2); print $2; exit}}')"
echo "GCC=$(gcc --version | head -n 1)"
echo "N={N}"
echo "STEPS={STEPS}"
echo "REPEATS={REPEATS}"
echo "--- modules ---"
module -t list 2>&1 || true
echo "=== END CONFIGURATION ==="

gcc -O3 -std=gnu11 -march=native m3s5_opt_lab.c -o m3s5_serial
gcc -O3 -std=gnu11 -march=native -pg m3s5_opt_lab.c -o m3s5_gprof

echo
echo "=== BASELINE REPEATS ==="
for rep in $(seq 1 {REPEATS}); do
    echo "RUN rep=$rep"
    ./m3s5_serial --mode bad --n {N} --steps {STEPS} \
        --checkpoint baseline_${{rep}}.bin
done

echo
echo "=== GNU TIME ==="
if [ -x /usr/bin/time ]; then
    /usr/bin/time -v ./m3s5_serial --mode bad --n {N} --steps {STEPS} \
        --checkpoint time_baseline.bin 2>&1 | sed 's/^/TIME /'
else
    echo "TIME unavailable"
fi

echo
echo "=== GPROF ==="
./m3s5_gprof --mode bad --n {N} --steps {STEPS} \
    --checkpoint gprof_baseline.bin >/dev/null
gprof ./m3s5_gprof gmon.out | head -n 45 || true

echo
echo "=== PERF OPTIONAL ==="
if command -v perf >/dev/null 2>&1; then
    perf stat -e cycles,instructions,cache-references,cache-misses \
        ./m3s5_serial --mode bad --n {N} --steps {STEPS} \
        --checkpoint perf_baseline.bin 2>&1 || true
else
    echo "perf unavailable"
fi

echo
echo "=== STRACE OPTIONAL: SMALL CASE ==="
if command -v strace >/dev/null 2>&1; then
    strace -c ./m3s5_serial --mode bad --n 256 --steps 10 \
        --checkpoint strace_baseline.bin 2>&1 || true
else
    echo "strace unavailable"
fi

echo "M3S5_BASELINE_DONE"
'''

Path("m3s5_baseline.slurm").write_text(baseline_script)
print(baseline_script)


In [ ]:
M3S5_BASE_JOB = submit("m3s5_baseline.slurm")
print("Job ID:", M3S5_BASE_JOB)


In [ ]:
wait_job(M3S5_BASE_JOB)
BASE_OUTPUT = show_output(M3S5_BASE_JOB, "m3s5_base")


## 5 — Read the evidence before editing

Answer before looking at the optimisation result:

1. Which phase dominates total runtime?
2. Which function dominates the `gprof` profile?
3. Is checkpoint I/O large enough to explain the runtime?
4. If `perf` worked, do the cache counters support investigating locality?
5. What is **measured evidence**, and what is still only a **hypothesis**?

The next cell extracts the repeated baseline timings.


In [ ]:
result_re = re.compile(
    r"RESULT mode=(\w+) workers=(\d+) n=(\d+) steps=(\d+) "
    r"total=([0-9.eE+-]+) compute=([0-9.eE+-]+) "
    r"checksum=([0-9.eE+-]+) io_ok=(\d+)"
)

def parse_results(text):
    rows = []
    for m in result_re.finditer(text or ""):
        mode, workers, n, steps, total, compute, checksum, io_ok = m.groups()
        rows.append({
            "mode": mode,
            "workers": int(workers),
            "n": int(n),
            "steps": int(steps),
            "total": float(total),
            "compute": float(compute),
            "checksum": float(checksum),
            "io_ok": int(io_ok),
        })
    return rows

base_rows = [
    r for r in parse_results(BASE_OUTPUT)
    if r["mode"] == "bad" and r["n"] == N and r["steps"] == STEPS
][:REPEATS]

base_median = statistics.median(r["total"] for r in base_rows)
base_compute = statistics.median(r["compute"] for r in base_rows)

print(f"Baseline runs: {len(base_rows)}")
for i, r in enumerate(base_rows, 1):
    print(
        f"run {i}: total={r['total']:.6f}s "
        f"compute={r['compute']:.6f}s checksum={r['checksum']:.6e}"
    )

print()
print(f"Median baseline total   : {base_median:.6f} s")
print(f"Median baseline compute : {base_compute:.6f} s")
print(f"Compute fraction        : {100*base_compute/base_median:.1f}%")


## 6 — State the hypothesis BEFORE changing code

A useful optimisation hypothesis has this form:

> **Evidence → suspected cause → specific change → predicted effect**

For this guided example:

> The compute phase dominates runtime and the baseline traversal steps through a row-major array by columns.  
> I hypothesize that poor spatial locality increases memory-system cost.  
> I will change only the loop order so the inner loop walks contiguous elements.  
> I predict lower compute time with the same checksum.

We do **not** change:

- numerical formula;
- input size;
- number of timesteps;
- compiler;
- `-O3`;
- Slurm allocation;
- repetition count;
- checkpoint size.

That is what makes the comparison interpretable.


## 7 — CHANGE ONE THING: loop order

The only experiment change is:

```text
--mode bad  →  --mode good
```

Everything else stays fixed.

### PREDICT

- Should the checksum change?
- Will total time necessarily fall by exactly the same proportion as compute time?
- If compute becomes much faster, could another phase become more visible?


In [ ]:
opt_script = f'''#!/bin/bash
#SBATCH --job-name=m3s5_opt
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=m3s5_opt-%j.out

set -euo pipefail
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== CONFIGURATION ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "HOST=$(hostname)"
echo "GCC=$(gcc --version | head -n 1)"
echo "N={N}"
echo "STEPS={STEPS}"
echo "REPEATS={REPEATS}"

gcc -O3 -std=gnu11 -march=native m3s5_opt_lab.c -o m3s5_serial
gcc -O3 -std=gnu11 -march=native -pg m3s5_opt_lab.c -o m3s5_gprof

echo
echo "=== OPTIMISED REPEATS ==="
for rep in $(seq 1 {REPEATS}); do
    echo "RUN rep=$rep"
    ./m3s5_serial --mode good --n {N} --steps {STEPS} \
        --checkpoint optimised_${{rep}}.bin
done

echo
echo "=== GPROF OPTIMISED ==="
./m3s5_gprof --mode good --n {N} --steps {STEPS} \
    --checkpoint gprof_optimised.bin >/dev/null
gprof ./m3s5_gprof gmon.out | head -n 45 || true

echo
echo "=== PERF OPTIONAL ==="
if command -v perf >/dev/null 2>&1; then
    perf stat -e cycles,instructions,cache-references,cache-misses \
        ./m3s5_serial --mode good --n {N} --steps {STEPS} \
        --checkpoint perf_optimised.bin 2>&1 || true
else
    echo "perf unavailable"
fi

echo "M3S5_OPT_DONE"
'''

Path("m3s5_optimised.slurm").write_text(opt_script)
print(opt_script)


In [ ]:
M3S5_OPT_JOB = submit("m3s5_optimised.slurm")
print("Job ID:", M3S5_OPT_JOB)


In [ ]:
wait_job(M3S5_OPT_JOB)
OPT_OUTPUT = show_output(M3S5_OPT_JOB, "m3s5_opt")


## 8 — Remeasure and decide

Compare **median with median** under controlled conditions.

We care about:

- correctness;
- total runtime;
- compute runtime;
- speedup;
- whether the profile changed in the predicted direction.


In [ ]:
opt_rows = [
    r for r in parse_results(OPT_OUTPUT)
    if r["mode"] == "good" and r["n"] == N and r["steps"] == STEPS
][:REPEATS]

opt_median = statistics.median(r["total"] for r in opt_rows)
opt_compute = statistics.median(r["compute"] for r in opt_rows)

base_checksum = statistics.median(r["checksum"] for r in base_rows)
opt_checksum = statistics.median(r["checksum"] for r in opt_rows)

speedup = base_median / opt_median
compute_speedup = base_compute / opt_compute

print("COMPARISON")
print(f"Baseline median total   : {base_median:.6f} s")
print(f"Optimised median total  : {opt_median:.6f} s")
print(f"Whole-program speedup   : {speedup:.2f}x")
print()
print(f"Baseline compute median : {base_compute:.6f} s")
print(f"Optimised compute median: {opt_compute:.6f} s")
print(f"Compute-phase speedup   : {compute_speedup:.2f}x")
print()
print(f"Baseline checksum       : {base_checksum:.12e}")
print(f"Optimised checksum      : {opt_checksum:.12e}")
print(f"Checksum difference     : {abs(base_checksum-opt_checksum):.3e}")

correct = abs(base_checksum-opt_checksum) < 1e-8 * max(1.0, abs(base_checksum))
print("Correctness:", "PASS" if correct else "CHECK")


## 9 — Connect back to scalability

An optimisation can change the way a program scales.

Now compile the **locality-friendly** implementation with OpenMP and run a small strong-scaling experiment:

```text
1 → 2 → 4 → 8 threads
```

The global problem size remains fixed.

This is not a second code optimisation: the code stays in `good` mode. We are now changing the **resource count** to examine scaling.

### PREDICT

Will 8 threads necessarily give 8× speedup over 1 thread?

What shared resource from M3.S2 may limit a memory-intensive kernel?


In [ ]:
scale_script = f'''#!/bin/bash
#SBATCH --job-name=m3s5_scale
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=m3s5_scale-%j.out

set -euo pipefail
cd "$SLURM_SUBMIT_DIR"

gcc -O3 -std=gnu11 -march=native -fopenmp m3s5_opt_lab.c -o m3s5_omp

for t in 1 2 4 8; do
    for rep in $(seq 1 {REPEATS}); do
        echo "SCALE rep=$rep threads=$t"
        OMP_NUM_THREADS=$t OMP_PROC_BIND=close OMP_PLACES=cores \
            ./m3s5_omp --mode good --n {N} --steps {STEPS} \
            --checkpoint scale_${{t}}_${{rep}}.bin
    done
done

echo "M3S5_SCALE_DONE"
'''

Path("m3s5_scaling.slurm").write_text(scale_script)
print(scale_script)


In [ ]:
M3S5_SCALE_JOB = submit("m3s5_scaling.slurm")
wait_job(M3S5_SCALE_JOB)
SCALE_OUTPUT = show_output(M3S5_SCALE_JOB, "m3s5_scale")


In [ ]:
scale_rows = [
    r for r in parse_results(SCALE_OUTPUT)
    if r["mode"] == "good" and r["n"] == N and r["steps"] == STEPS
]

by_threads = {}
for r in scale_rows:
    by_threads.setdefault(r["workers"], []).append(r["total"])

scale_summary = []
for t in sorted(by_threads):
    scale_summary.append((t, statistics.median(by_threads[t])))

t1 = dict(scale_summary)[1]

print(f'{"threads":>8} {"median(s)":>12} {"speedup":>10} {"efficiency":>12}')
for t, med in scale_summary:
    s = t1 / med
    e = 100*s/t
    print(f"{t:8d} {med:12.6f} {s:10.2f} {e:11.1f}%")


## 10 — I/O, libraries and reproducibility: choose the next question

The profile tells you **where to investigate**. It does not automatically tell you the fix.

| Evidence points to... | Next question / possible direction |
|---|---|
| Poor memory locality | Can loop order or data layout improve access? |
| Standard FFT / BLAS-like hotspot | Is a mature optimized library appropriate? |
| I/O / many files | Can the output pattern or file structure change? |
| Synchronization / waiting | Is work imbalanced or communication too frequent? |
| Strong-scaling flattening | Is overhead dominating useful work? |

The checkpoint phase also demonstrates an important S14 idea:

> If compute becomes faster while I/O time stays constant, I/O becomes a larger fraction of runtime.

So an optimisation can **move the bottleneck**.


## 11 — Build the minimum evidence bundle

A trustworthy result records:

**CODE / BUILD**
- source version;
- compiler;
- optimisation flags.

**INPUT**
- N;
- timesteps.

**ENVIRONMENT**
- modules;
- host;
- Slurm resources.

**MEASUREMENT**
- repetitions;
- timer boundaries;
- statistic used.

**OUTPUT**
- checksum;
- raw job output;
- result/checkpoint files.

The next cell creates a small manifest.


In [ ]:
manifest = {
    "session": "M3.S5 Practice 3 guided lab",
    "question": "Does locality-friendly loop order reduce runtime under controlled conditions?",
    "input": {"n": N, "steps": STEPS},
    "repetitions": REPEATS,
    "baseline": {
        "mode": "bad",
        "median_total_s": base_median,
        "median_compute_s": base_compute,
        "job_id": str(globals().get("M3S5_BASE_JOB", "")),
    },
    "changed": {
        "mode": "good",
        "median_total_s": opt_median,
        "median_compute_s": opt_compute,
        "job_id": str(globals().get("M3S5_OPT_JOB", "")),
    },
    "speedup": speedup,
    "checksum_difference": abs(base_checksum-opt_checksum),
    "scaling_job_id": str(globals().get("M3S5_SCALE_JOB", "")),
}

Path("m3s5_run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(Path("m3s5_run_manifest.json").read_text())


## 12 — Final reflection

Answer before moving to Part 2.

1. Why are repeated baseline measurements stronger evidence than one timing?
2. Which phase/function was the main hotspot?
3. What did the locality hypothesis predict?
4. What was the **one factor** changed?
5. Did total runtime improve? By how much?
6. How did you verify correctness after the change?
7. Why did 8 threads not necessarily give 8× speedup?
8. Name four pieces of information needed to reproduce the experiment.
9. If the profile had shown I/O dominating instead, would loop interchange have been the right optimisation?

<details>
<summary><strong>Key idea</strong></summary>

The goal is not to make *something* faster.

The goal is to make an evidence-supported change and then prove what happened under comparable conditions.

</details>


# Part 1 complete

You have completed one full Module 3 evidence chain:

```text
CORRECTNESS
    ↓
BASELINE
    ↓
PROFILE
    ↓
HYPOTHESIS
    ↓
CHANGE ONE THING
    ↓
REMEASURE
    ↓
COMPARE
    ↓
DOCUMENT
```

## Next — Part 2: Practice 3 Individual Assignment

Part 2 will be completed independently through **SSH + Slurm**.

You will **not simply repeat the loop-order example**.

Blackboard will define the independent task, required evidence bundle, interpretation and submission rules.

The same reasoning process will be required:

> **baseline → profile → hypothesis → one change → remeasure → explain → reproduce**
